## Example Gene

In [1]:
import pandas as pd
import numpy as np
import igraph as ig
from PathNet import *

In [2]:
gene_expr = pd.read_csv('data/data_raw/HiSeqV2', sep='\t', index_col=0).transpose()
gene_net = pd.read_csv('data/data_raw/HomoSapiens_lcb_hq.txt', sep='\t')
mirna_expr = pd.read_csv('data/data_raw/TCGA-mirna_expr.csv', index_col=0)

# we only need the first tumor sample
mirna_expr.index = mirna_expr.index.str.slice(stop=-1)
mirna_expr = mirna_expr[~mirna_expr.index.duplicated(keep="first")]
mirna_expr = mirna_expr[mirna_expr.index.str.endswith("-01")]

# find common samples
samples_in_gene = gene_expr.index.values
samples_in_mirna = mirna_expr.index.values

common_samples = np.intersect1d(samples_in_gene, samples_in_mirna)

# make sure they align
gene_expr = gene_expr.loc[common_samples]
mirna_expr = mirna_expr.loc[gene_expr.index]

In [ ]:
MIN_SAMPLE_PCT = 0.05  # lowest expression percentage

def filter_low_expr(expr_df, min_sample_pct=MIN_SAMPLE_PCT):
    n_samples = expr_df.shape[0]
    min_samples = int(n_samples * min_sample_pct)
    expr_count = (expr_df > 0).sum(axis=0)
    high_expr = expr_count[expr_count >= min_samples].index
    return expr_df[high_expr]

gene_expr_filtered = filter_low_expr(gene_expr, min_sample_pct=0.2)
mirna_expr_filtered = filter_low_expr(mirna_expr)

print(f"Gene: {gene_expr.shape[1]} → {gene_expr_filtered.shape[1]}")
print(f"miRNA: {mirna_expr.shape[1]} → {mirna_expr_filtered.shape[1]}")

gene_expr = gene_expr_filtered
mirna_expr = mirna_expr_filtered

基因过滤: 20530 → 18625
miRNA过滤: 632 → 334


In [4]:
interactions = set()
for _, row in gene_net.iterrows():
    gene_a, gene_b = row["Gene_A"], row["Gene_B"]
    interactions.add(frozenset({gene_a, gene_b})) 

genes_in_expression = set(gene_expr.columns)

In [ ]:
valid_pairs = []
for pair in interactions:
    if len(pair) == 1:
        continue  # when no pair is available
    gene_a, gene_b = pair
    if gene_a in genes_in_expression and gene_b in genes_in_expression:
        valid_pairs.append((gene_a, gene_b))

valid_pairs_df = pd.DataFrame(valid_pairs, columns=["Gene_A", "Gene_B"])
print(f"There are {len(valid_pairs_df)} pairs of interacting genes.")
valid_pairs_df.head()

There are 30538 pairs of interacting genes.


,Gene_A,Gene_B
0,PPP3R1,PPP3CB
1,SMCHD1,SUMO2
2,HDAC1,E2F1
3,MAP1LC3B,NEDD4
4,DNAJC3,EIF2AK2


In [6]:
genes_in_network = set(valid_pairs_df["Gene_A"]).union(set(valid_pairs_df["Gene_B"]))
print(f"There are {len(genes_in_network)} genes in the network.")

filtered_expression = gene_expr.loc[:, gene_expr.columns.isin(genes_in_network)]

print(f"After filtering, we maintain {len(filtered_expression.columns)} genes, while originally we have {len(gene_expr.columns)} genes.")
filtered_expression.head()

There are 8523 genes in the network.
After filtering, we maintain 8523 genes, while originally we have 18625 genes.


sample,HIF3A,RNF10,RNF11,RNF13,FGFR1OP2,ATRX,PMM2,NCBP1,RBM14,NCBP2,...,NFIC,NFIB,NFIX,PLEKHG7,SLC7A14,SLC7A10,GNGT1,TULP3,BCL6B,SELP
TCGA-3C-AAAU-01,2.3606,11.6317,11.2053,10.1911,8.3700,11.7001,9.8610,9.8876,10.7770,10.5727,...,11.1381,7.4142,12.4178,1.2501,0.0,0.7594,0.0000,8.9560,7.8250,5.6018
TCGA-3C-AALI-01,0.6265,11.8757,10.4259,10.1674,8.6715,9.7384,10.2038,9.9498,10.7335,10.8880,...,9.2242,11.0613,9.0823,7.2024,0.0,3.5174,3.2780,9.3888,8.1093,5.5942
TCGA-3C-AALJ-01,0.9310,12.2616,10.7842,9.5713,8.5936,9.9302,11.1468,10.0473,10.1364,10.5513,...,9.6053,9.5609,8.8189,1.4922,0.0,3.7848,1.4922,9.3284,8.7597,5.6941
TCGA-3C-AALK-01,2.6732,11.9567,11.0071,10.0354,8.4158,10.5079,10.3365,9.9018,10.4616,10.6503,...,9.9088,11.1114,10.7099,2.4728,0.0,0.8796,0.0000,9.3354,7.9614,8.6253
TCGA-4H-AAAK-01,2.2720,11.7566,10.8499,9.9325,8.5128,10.5833,9.9119,9.9054,10.1024,11.0322,...,9.6045,10.9942,11.6683,1.8291,0.0,1.1896,0.0000,9.0038,7.6902,5.7256


In [ ]:
gene_expr = filtered_expression

mirna_df = pd.read_csv("data/data_processed/miRNA_gene_mapping.csv")
mirna_df['miRNA'] = mirna_df['miRNA'].str.lower()
mirna_df.set_index('miRNA', inplace=True)
mirna_df = mirna_df.loc[mirna_expr.columns]
mirna_df.reset_index(inplace=True)

target_genes = set()
for _, row in mirna_df.iterrows():
    if pd.notna(row["target_genes"]): 
        genes = row["target_genes"].strip("[]").replace("'", "").replace('"', "").split(", ")
        genes = [g.strip() for g in genes if g.strip()]  
        target_genes.update(genes)
        mirna_df.at[_, "target_genes"] = genes  

print(f"There are {len(target_genes)} miRNAs.")

gene_to_idx = {gene: idx for idx, gene in enumerate(gene_expr.columns)}
mirna_to_idx = {mirna: idx for idx, mirna in enumerate(mirna_expr.columns)}

共找到 14417 个唯一 miRNA 靶基因。


In [ ]:
from scipy.sparse import lil_matrix

n_genes = len(gene_to_idx)
n_mirnas = len(mirna_to_idx)

reg_matrix = lil_matrix((n_genes, n_mirnas), dtype=np.float32)

for _, row in mirna_df.iterrows():
    mirna_idx = mirna_to_idx[row["index"]]
    for gene in row["target_genes"]:
        if gene in gene_to_idx:
            gene_idx = gene_to_idx[gene]
            reg_matrix[gene_idx, mirna_idx] = 1

In [9]:
n_genes = len(gene_expr.columns)

# gene_gene relationship
adj_matrix = lil_matrix((n_genes, n_genes), dtype=np.int8)

for _, row in valid_pairs_df.iterrows():
    i, j = gene_to_idx[row["Gene_A"]], gene_to_idx[row["Gene_B"]]
    # I believe this is fine, for it will only be used as a reference?
    adj_matrix[i, j] = adj_matrix[j, i] = 1

In [10]:
clinical_outcome = pd.read_csv('data/labels/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix', index_col=0, sep='\t')
clinical_outcome.loc[gene_expr.index, :]
clinical_outcome = clinical_outcome.loc[gene_expr.index, :]
clinical_outcome.head()

,AJCC_Stage_nature2012,Age_at_Initial_Pathologic_Diagnosis_nature2012,CN_Clusters_nature2012,Converted_Stage_nature2012,Days_to_Date_of_Last_Contact_nature2012,Days_to_date_of_Death_nature2012,ER_Status_nature2012,Gender_nature2012,HER2_Final_Status_nature2012,Integrated_Clusters_no_exp__nature2012,...,_GENOMIC_ID_TCGA_BRCA_mutation_wustl_gene,_GENOMIC_ID_TCGA_BRCA_miRNA_GA,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2_percentile,_GENOMIC_ID_data/public/TCGA/BRCA/miRNA_GA_gene,_GENOMIC_ID_TCGA_BRCA_gistic2thd,_GENOMIC_ID_data/public/TCGA/BRCA/miRNA_HiSeq_gene,_GENOMIC_ID_TCGA_BRCA_G4502A_07_3,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2,_GENOMIC_ID_TCGA_BRCA_gistic2,_GENOMIC_ID_TCGA_BRCA_PDMarray
TCGA-3C-AAAU-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,6ef883fc-81f3-4089-95e0-86904ffc0d38,NaN,TCGA-3C-AAAU-01A-11D-A41E-01,TCGA-3C-AAAU-01,NaN,6ef883fc-81f3-4089-95e0-86904ffc0d38,TCGA-3C-AAAU-01A-11D-A41E-01,NaN
TCGA-3C-AALI-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,dd8d3665-ec9d-45be-b7b9-a85dac3585e2,NaN,TCGA-3C-AALI-01A-11D-A41E-01,TCGA-3C-AALI-01,NaN,dd8d3665-ec9d-45be-b7b9-a85dac3585e2,TCGA-3C-AALI-01A-11D-A41E-01,NaN
TCGA-3C-AALJ-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,c924c2a8-ab41-4499-bb30-79705cc17d45,NaN,TCGA-3C-AALJ-01A-31D-A41E-01,TCGA-3C-AALJ-01,NaN,c924c2a8-ab41-4499-bb30-79705cc17d45,TCGA-3C-AALJ-01A-31D-A41E-01,NaN
TCGA-3C-AALK-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1a19e068-d213-41ef-aebb-104017a883cc,NaN,TCGA-3C-AALK-01A-11D-A41E-01,TCGA-3C-AALK-01,NaN,1a19e068-d213-41ef-aebb-104017a883cc,TCGA-3C-AALK-01A-11D-A41E-01,NaN
TCGA-4H-AAAK-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2ea9e472-a408-4ae0-975d-50a566f22b2a,NaN,TCGA-4H-AAAK-01A-12D-A41E-01,TCGA-4H-AAAK-01,NaN,2ea9e472-a408-4ae0-975d-50a566f22b2a,TCGA-4H-AAAK-01A-12D-A41E-01,NaN


In [11]:
# only want those with labels
valid_index = clinical_outcome.index[clinical_outcome['ER_Status_nature2012'].notna()]
gene_expr = gene_expr.loc[valid_index, :]
mirna_expr = mirna_expr.loc[valid_index, :]
clinical_outcome = clinical_outcome.loc[valid_index, ['ER_Status_nature2012']]

labels = np.zeros(len(gene_expr.index))
labels[clinical_outcome['ER_Status_nature2012'] == 'Positive'] = 1

In [12]:
maximum_step = 2
num_hidden_layer_neuron_list = []
feature_meta = reg_matrix.transpose()
expression = gene_expr.values
mirna_expr = mirna_expr.values
target_genes = ['OPRK1', 'CAMKV', 'MAGEB4', 'CNGB1', 'TLX3', 'SCEL', 'NR2E1',
       'CCKBR', 'FETUB', 'KCNJ3', 'LIN28B', 'PDX1', 'PRKAG3', 'GHRHR',
       'TRIM10', 'FOXG1', 'GSTA5', 'MAGEA4', 'AHSG', 'TUBA3C', 'PYY',
       'ALPP', 'SERPINB4', 'C1orf94', 'GPR139', 'TEKT1', 'GHRH', 'PSKH2']
labels = labels
drop_out = 0.1
sub_graph = ig.Graph.Adjacency((adj_matrix > 0).toarray().tolist(), mode='undirected')  # 将邻接矩阵转为布尔值
sub_graph.vs["name"] = gene_expr.columns.values

In [13]:
lr = 1e-3
weight_decay = 0
num_epoch = 100
random_seed = 2781
batch_size = 16
model, partition_mtx_dict, connectionList = sparse_nn(expression=expression, labels=labels, target_feature=target_genes, feature_meta=feature_meta, knowledge_graph=sub_graph, random_seed=random_seed, maximum_step=maximum_step, meta=False, mirna_expr=mirna_expr,
          num_hidden_layer_neuron_list=num_hidden_layer_neuron_list, drop_out=drop_out, batch_size=batch_size, lr=lr, weight_decay=weight_decay, num_epoch=num_epoch)

epoch: 1, test acc: 0.834061, train acc: 0.803002
epoch: 2, test acc: 0.868996, train acc: 0.851782
epoch: 3, test acc: 0.899563, train acc: 0.879925
epoch: 4, test acc: 0.908297, train acc: 0.906191
epoch: 5, test acc: 0.912664, train acc: 0.908068


KeyboardInterrupt: 

In [ ]:
def save_param(model, connectionlist,
               path=f'result/geneparams_{random_seed}.npz'):
    np.savez_compressed(path,
        model_params=model,
        connectionlist={str(k):v for k,v in enumerate(connectionlist)}
                        )
    return

In [ ]:
save_param(model, connectionList)